Hands-on the exported AppWorld packet. Change `TREE_ID`, re-cut, print-walk.

In [8]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

from orchard import Orchard, build_dynamic_cut, validate_dynamic_cut, walk_cut_json
from orchard.cuts import get_optimal_cut_labels
from orchard.tree import default_node_label

VIEW = Path.cwd() if (Path.cwd() / "artifacts" / "appworld").exists() else Path.cwd() / "orchard-view"
sys.path.insert(0, str(VIEW / "scripts"))
from export_appworld_orchard import CUT_PARAMS, TREE_SPECS, align_feature_matrix  # noqa: E402

PACKET = VIEW / "artifacts" / "appworld"
#TREE_ID = "domain"  # or "function"
TREE_ID = "function"

orchard = Orchard.load(PACKET / "orchard")
cuts = {
    name: json.loads((PACKET / "cuts" / f"{name}.calinski_optimal.json").read_text(encoding="utf-8"))
    for name in ("domain", "function")
}
cut_manifest = json.loads((PACKET / "cuts" / "manifest.json").read_text(encoding="utf-8"))

tree = orchard.tree(TREE_ID)
cut = cuts[TREE_ID]
spec = next(row for row in TREE_SPECS if row["tree_id"] == TREE_ID)
archive = np.load(spec["npz"])
matrix = align_feature_matrix(
    archive["matrix"],
    [str(tid) for tid in archive["tool_ids"]],
    list(tree.item_ids),
    TREE_ID,
)
print(TREE_ID, "docs", len(orchard.documents), "leaves", tree.leaf_count, "matrix", matrix.shape)

function docs 457 leaves 457 matrix (457, 457)


In [9]:
doc = orchard.documents[0]
print("doc", doc.item_id, "source", doc.source, "title", doc.title)
print("root", tree.root_node_id)
print("root label", tree.label_for(tree.root_node_id))
print("active labels", tree.active_label_set, len(tree.labels[tree.active_label_set]))
print("cut", cut["top_criterion"], cut["cut_optimizer"], "k", cut["top_partition_count"], "warnings", cut["warnings"])
print("diag", float(np.min(np.diag(matrix))), float(np.max(np.diag(matrix))))
print("CUT_PARAMS", CUT_PARAMS)

doc amazon.add_address source appworld title amazon.add_address
root member_fa955a31c93e0faf46e743098371cfff7a6519c28c4a43755e04d435614d6b02
root label function
active labels phase5a_intrinsic 913
cut optimal calinski_harabasz_score k 3 warnings []
diag 1.0 1.0
CUT_PARAMS {'top_criterion': 'optimal', 'cut_optimizer': 'calinski_harabasz_score', 'cut_polarity': 1, 'min_width': 3, 'max_width': 10, 'target_width': None, 'max_depth': 5, 'threshold_steps': 32}


In [10]:
def walk_cut(
    node: dict,
    tree,
    *,
    depth: int = 0,
    max_depth: int = 1,
    show_leaf: bool = True,
    show_kind: bool = True,
) -> None:
    cid = node.get("canonical_node_id") or node.get("node_id")
    src = tree.node(cid)
    kind = f"{src['kind']}: " if show_kind else ""
    if src["kind"] == "leaf":
        label = src.get("item_id")
    else:
        label = tree.label_for(cid) or default_node_label(tree, cid)
    n = src["descendant_count"]
    print("  " * depth + f"{kind}{str(label or cid)[:90]}  ({n})")
    if show_leaf:
        for leaf_cid in node.get("direct_leaf_node_ids") or []:
            leaf = tree.node(leaf_cid)
            leaf_kind = "leaf: " if show_kind else ""
            print("  " * (depth + 1) + f"{leaf_kind}{leaf.get('item_id') or leaf_cid}")
    if depth >= max_depth:
        return
    for child in node.get("children") or []:
        walk_cut(
            child,
            tree,
            depth=depth + 1,
            max_depth=max_depth,
            show_leaf=show_leaf,
            show_kind=show_kind,
        )

In [11]:
walk_cut(cut["root"], tree, max_depth=1, show_leaf=False, show_kind=False)

function  (457)
  Unified authenticated data update and deletion  (234)
  Authentication and code dispatch  (61)
  Search-Retrieve Unified Control  (162)


In [7]:
# unconstrained flat sweep (k=2..17) vs dynamic window (3..10)
Z = tree.linkage
for name, metric, polarity in (
    ("silhouette", silhouette_score, 1),
    ("calinski_harabasz", calinski_harabasz_score, 1),
    ("davies_bouldin", davies_bouldin_score, -1),
):
    _, k_flat = get_optimal_cut_labels(
        Z, matrix, metric=metric, polarity=polarity, min_clusters=2, max_clusters=17
    )
    _, k_win = get_optimal_cut_labels(
        Z, matrix, metric=metric, polarity=polarity, min_clusters=3, max_clusters=10
    )
    print(f"{name}: flat k={k_flat}  windowed k={k_win}")
print("shipped dynamic top_partition_count", cut["top_partition_count"])

silhouette: flat k=11  windowed k=10
calinski_harabasz: flat k=11  windowed k=9
davies_bouldin: flat k=10  windowed k=10
shipped dynamic top_partition_count 9


In [ ]:
rebuilt = build_dynamic_cut(
    tree,
    top_criterion="optimal",
    cut_optimizer="calinski_harabasz_score",
    feature_matrix=matrix,
    cut_polarity=1,
    min_width=3,
    max_width=10,
    target_width=None,
    max_depth=5,
    threshold_steps=32,
)
validate_dynamic_cut(rebuilt, tree)
print("rebuilt top_partition_count", rebuilt["top_partition_count"], "warnings", rebuilt["warnings"])
walk_cut(rebuilt["root"], tree, max_depth=1, show_leaf=False, show_kind=False)

In [ ]:
# orchard walk_cut_json promotes direct_leaf_node_ids into children — optional viewer shape
walk_cut_json(rebuilt, tree, max_depth=1)